# 鸟的分类


## 1. 项目背景介绍

鸟类是地球上最为多样和迷人的生物之一。它们属于脊椎动物门鸟纲，具有羽毛、嘴喙和两只前肢改造为翅膀的特征。鸟类在生态系统中扮演着重要的角色，包括传粉、种子散布和控制害虫等功能。

鉴于鸟类的多样性和生态重要性，本项目旨在利用机器学习工具和算法对鸟类进行分类和研究。  
通过应用机器学习技术，可以提高鸟类识别的自动化水平，为广大鸟类爱好者提供一个便捷的工具，帮助他们更好地了解和欣赏鸟类世界，同时有助于鸟类学家和生物多样性研究人员更好地理解和保护我们共享的自然遗产。

- 文件列表
  
  - 包含 200 个子文件为 200 类共 8872 张鸟类图像，每个文件夹包含一定数量 JPG 格式的图片，图片命名格式为：类名_编号。

下面以训练集数据为例说明：

- 数据集的整体特征

| 数据集名称   | 数据类型 | 种类数 | 实例数 | 值缺失 | 相关任务   |
| :----------- | :------- | :----- | :----- | :----- | :--------- |
| 鸟类图片 | 图像数据 | 200     | 8872   | 无     | 分类 |

## 2. 数据列表生成和参数配置

- 导入相关库  

In [1]:
import os
import random
import torch
import numpy as np
import matplotlib.pyplot as plt
import PIL.Image as Image
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torchvision.transforms as transforms
from torch.optim import Adam
from sklearn.metrics import accuracy_score

seed = 1234
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

Duplicate key in file PosixPath('/usr/local/lib/python3.9/dist-packages/matplotlib/mpl-data/matplotlibrc'), line 801 ('font.family: sans-serif')
Duplicate key in file PosixPath('/usr/local/lib/python3.9/dist-packages/matplotlib/mpl-data/matplotlibrc'), line 802 ('font.sans-serif: SimHei')


- 参数配置

In [2]:
train_parameters = {
    "input_size": [3, 224, 224],                                #输入图片的shape
    "class_dim": 200,                                          #分类数
    "target_path":"/home/jovyan/work/datasets/689320899e78856c69427ad2-momodel/bird",                     #要解压的路径
    "train_list_path": "/home/jovyan/work/train.txt",       #train.txt路径
    "eval_list_path": "/home/jovyan/work/eval.txt",         #eval.txt路径
    "label_dict":{},         #标签字典
    "num_epochs": 40,                                         #训练轮数
    "train_batch_size": 2,                                   #训练时每个批次的大小
    "learning_strategy": {                                    #优化函数相关的配置
        "lr": 0.0001                                           #超参数学习率
    }, 
    'skip_steps': 5,                                         #每N个批次打印一次结果
    'save_steps': 10,                                         #每N个批次保存一次模型参数
    "checkpoints": "/home/jovyan/work/results"          #保存的路径

}

- 将数据进行统计并分为训练集和验证集

In [3]:
def get_train_data_list(target_path,train_list_path,eval_list_path):
    '''
    生成数据列表
    '''
    #获取所有类别保存的文件夹名称
    data_list_path=target_path
    class_dirs = os.listdir(data_list_path)
    #总的图像数量
    all_class_images = 0
    #存放类别标签
    class_label=0
    #存储要写进eval.txt和train.txt中的内容
    trainer_list=[]
    eval_list=[]
    class_dim=0
    #读取每个类别
    for class_dir in class_dirs:
        class_dim += 1
        #统计每个类别有多少张图片
        class_sum = 0
        #获取类别路径 
        path = os.path.join(data_list_path,class_dir)
        # 获取所有图片
        img_paths = os.listdir(path)
        for img_path in img_paths:                                  # 遍历文件夹下的每个图片
            name_path = os.path.join(path,img_path)                       # 每张图片的路径
            if class_sum % 12 == 0:                                 # 每10张图片取一个做验证数据
                eval_list.append(name_path + "\t%d" % class_label + "\n")
            else:
                trainer_list.append(name_path + "\t%d" % class_label + "\n")
            class_sum += 1                                          #每类图片的数目
        
        #初始化标签列表
        train_parameters['label_dict'][str(class_label)] = class_dir[4:]
        class_label += 1
        
    #初始化分类数
    train_parameters['class_dim'] = class_dim
    print(train_parameters)
    #乱序  
    random.shuffle(eval_list)
    with open(eval_list_path, 'a') as f:
        for eval_image in eval_list:
            f.write(eval_image) 
    #乱序        
    random.shuffle(trainer_list) 
    with open(train_list_path, 'a') as f2:
        for train_image in trainer_list:
            f2.write(train_image) 

    print ('生成训练数据列表完成！')

In [4]:
#参数初始化
train_target_path=train_parameters['target_path']
train_list_path=train_parameters['train_list_path']
eval_list_path=train_parameters['eval_list_path']

#每次生成数据列表前，首先清空train.txt和eval.txt
with open(train_list_path, 'w') as f: 
    f.seek(0)
    f.truncate() 
with open(eval_list_path, 'w') as f: 
    f.seek(0)
    f.truncate()    

#生成数据列表   
get_train_data_list(train_target_path,train_list_path,eval_list_path)

{'input_size': [3, 224, 224], 'class_dim': 200, 'target_path': '/home/jovyan/work/datasets/689320899e78856c69427ad2-momodel/bird', 'train_list_path': '/home/jovyan/work/train.txt', 'eval_list_path': '/home/jovyan/work/eval.txt', 'label_dict': {'0': 'Black_footed_Albatross', '1': 'Laysan_Albatross', '2': 'Sooty_Albatross', '3': 'Groove_billed_Ani', '4': 'Crested_Auklet', '5': 'Least_Auklet', '6': 'Parakeet_Auklet', '7': 'Rhinoceros_Auklet', '8': 'Brewer_Blackbird', '9': 'Red_winged_Blackbird', '10': 'Rusty_Blackbird', '11': 'Yellow_headed_Blackbird', '12': 'Bobolink', '13': 'Indigo_Bunting', '14': 'Lazuli_Bunting', '15': 'Painted_Bunting', '16': 'Cardinal', '17': 'Spotted_Catbird', '18': 'Gray_Catbird', '19': 'Yellow_breasted_Chat', '20': 'Eastern_Towhee', '21': 'Chuck_will_Widow', '22': 'Brandt_Cormorant', '23': 'Red_faced_Cormorant', '24': 'Pelagic_Cormorant', '25': 'Bronzed_Cowbird', '26': 'Shiny_Cowbird', '27': 'Brown_Creeper', '28': 'American_Crow', '29': 'Fish_Crow', '30': 'Black_

## 3. 定义并测试数据集

- 定义数据集类，对图像进行统一格式处理。转化为 RGB 三通道，并设置大小为 224 * 224

In [5]:
class BirdDataset(Dataset):
    def __init__(self, data_path, mode='train', transform=None):
        """
        数据读取器
        :param data_path: 数据集所在路径
        :param mode: train or eval
        """
        super(BirdDataset, self).__init__()
        self.data_path = data_path
        self.img_paths = []
        self.labels = []
        self.transform = transform

        if mode == 'train':
            with open(os.path.join(self.data_path, "train.txt"), "r", encoding="utf-8") as f:
                self.info = f.readlines()
            for img_info in self.info:
                img_path, label = img_info.strip().split('\t')
                self.img_paths.append(img_path)
                self.labels.append(int(label))

        else:
            with open(os.path.join(self.data_path, "eval.txt"), "r", encoding="utf-8") as f:
                self.info = f.readlines()
            for img_info in self.info:
                img_path, label = img_info.strip().split('\t')
                self.img_paths.append(img_path)
                self.labels.append(int(label))

    def __getitem__(self, index):
        """
        获取一组数据
        :param index: 文件索引号
        :return:
        """
        # 第一步打开图像文件并获取label值
        img_path = self.img_paths[index]
        img = Image.open(img_path)
        if img.mode != 'RGB':
            img = img.convert('RGB') 
        img = self.transform(img)
        label = self.labels[index]
        label = np.array([label], dtype="int64")
        return img, label

    def print_sample(self, index: int = 0):
        print("文件名", self.img_paths[index], "\t标签值", self.labels[index])

    def __len__(self):
        return len(self.img_paths)

- 定义数据增强

In [6]:
train_transform = transforms.Compose([
    transforms.Resize(256), 
                        transforms.RandomRotation(degrees=15),  # 图像以-15到15的角度随机旋转
                        transforms.RandomHorizontalFlip(),     # 随机水平旋转图像，默认概率为50%
                        transforms.CenterCrop(224),   # 将图片从中心切剪成3*224*224大小的图片
                        transforms.ToTensor(),         # 把图片进行归一化为0-1，并把数据转换成Tensor类型
                        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
                        ])

eval_transform = transforms.Compose([
    transforms.Resize((224,224)), 
                        transforms.ToTensor(),         # 把图片进行归一化为0-1，并把数据转换成Tensor类型
                        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
                        ])

- 分别生成训练集和验证集并打印部分数据进行测试

In [7]:
#训练数据加载
train_dataset = BirdDataset('/home/jovyan/work/',mode='train', transform = train_transform)
train_loader = DataLoader(train_dataset, 
                          batch_size=train_parameters['train_batch_size'], 
                          shuffle=True
                                    )
#测试数据加载
eval_dataset = BirdDataset('/home/jovyan/work/',mode='eval', transform = eval_transform)
eval_loader = DataLoader(eval_dataset,
                                   batch_size=train_parameters['train_batch_size'], 
                                   shuffle=False
                                   )

In [8]:
train_dataset.print_sample(22)
print(train_dataset.__len__())
eval_dataset.print_sample(10)
print(eval_dataset.__len__())
print(eval_dataset.__getitem__(10)[0].shape)
print(eval_dataset.__getitem__(10)[1].shape)

文件名 /home/jovyan/work/datasets/689320899e78856c69427ad2-momodel/bird/114.Black_throated_Sparrow/Black_Throated_Sparrow_0080_107050.jpg 	标签值 113
8076
文件名 /home/jovyan/work/datasets/689320899e78856c69427ad2-momodel/bird/176.Prairie_Warbler/Prairie_Warbler_0123_172603.jpg 	标签值 175
796
torch.Size([3, 224, 224])
(1,)


## 4. 定义卷积神经网络

- 定义卷积神经网络，采用四层卷积层和四层池化层交叉以及五层全连接层，为了保证模型更好的效果，这里设置了较大的参数量来学习更多的内容。

In [9]:
#自行定义卷积网络
class MyCNN(nn.Module):
    def __init__(self, class_nums):
        super(MyCNN, self).__init__()
        self.hidden1 = nn.Conv2d(in_channels = 3, out_channels = 64, kernel_size = 3, stride = 1, padding = 1)
        self.hidden2 = nn.MaxPool2d(kernel_size = 2, stride = 2)
        self.hidden3 = nn.Conv2d(in_channels = 64, out_channels = 128, kernel_size = 3, stride = 1, padding = 1)
        self.hidden4 = nn.MaxPool2d(kernel_size = 2, stride = 2)
        self.hidden5 = nn.Conv2d(in_channels = 128, out_channels = 256, kernel_size = 3, stride = 1, padding = 1)
        self.hidden6 = nn.MaxPool2d(kernel_size = 2, stride = 2)
        self.hidden7 = nn.Conv2d(in_channels = 256, out_channels = 512, kernel_size = 3, stride = 1, padding = 1)
        self.hidden8 = nn.MaxPool2d(kernel_size = 2, stride = 2)
        self.hidden11 = nn.Linear(512 * 14 * 14, 2048)
        self.hidden12 = nn.Linear(2048, 1024)
        self.hidden13 = nn.Linear(1024, 512)
        self.hidden14 = nn.Linear(512, 256)
        self.hidden15 = nn.Linear(256, class_nums)
        self.relu = nn.ReLU()
        self.flatten = nn.Flatten()
        
    def forward(self, input):
        x = self.hidden1(input)
        x = self.relu(x)

        x = self.hidden2(x)
    
        x = self.hidden3(x)
        x = self.relu(x)

        x = self.hidden4(x)

        x = self.hidden5(x)
        x = self.relu(x)

        x = self.hidden6(x)

        x = self.hidden7(x)
        x = self.relu(x)

        x = self.hidden8(x)

        x = self.flatten(x)

        x = self.hidden11(x)
        x = self.relu(x)
        
        x = self.hidden12(x)
        x = self.relu(x)
        
        x = self.hidden13(x)
        x = self.relu(x)
        
        x = self.hidden14(x)
        x = self.relu(x)

        out = self.hidden15(x)
        return out

## 5. 模型训练

In [10]:
def draw_process(title,color,iters,data,label):
    plt.title(title, fontsize=24)
    plt.xlabel("iter", fontsize=20)
    plt.ylabel(label, fontsize=20)
    plt.plot(iters, data,color=color,label=label) 
    plt.legend()
    plt.grid()
    plt.show()

- 使用交叉熵作为损失函数
- 使用 Adam 作为优化器
- 编写训练过程，在训练时打印中间过程并报保存模型参数， 在 GPU job 中进行训练。

In [ ]:
use_gpu = torch.cuda.is_available()
model = MyCNN(train_parameters['class_dim'])

if use_gpu:
    model = model.cuda()
cross_entropy = nn.CrossEntropyLoss()
optimizer = Adam(params=model.parameters(), lr=train_parameters['learning_strategy']['lr']) 
best_eval_acc = 0
save_path = train_parameters["checkpoints"]+"/save_eval_best.ckpt"
                                  


In [ ]:
steps = 0
Iters, total_loss, total_acc = [], [], []

for epo in range(train_parameters['num_epochs']):
    model.train()
    for _, data in enumerate(train_loader):
        steps += 1
        x_data = data[0]
        y_data = data[1].view(-1)
        if use_gpu:
            x_data, y_data = x_data.cuda(), y_data.cuda()
        predicts = model(x_data)
        loss = cross_entropy(predicts, y_data)
        pred = predicts.cpu().argmax(axis=1)
        acc = accuracy_score(pred, y_data.cpu())
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        if steps % train_parameters["skip_steps"] == 0:
            Iters.append(steps)
            total_loss.append(loss.cpu().item())
            total_acc.append(acc)
            #打印中间过程
            print('epo: {}, step: {}, loss is: {}, acc is: {}'\
                  .format(epo, steps, loss.cpu().item(), acc))
        #保存模型参数
    
    model.eval()
    eval_acc = []
    with torch.no_grad():
        for _, data in enumerate(eval_loader):
            x_data = data[0]
            y_data = data[1].view(-1)
            predicts = model(x_data)
            pred = predicts.cpu().argmax(axis=1)
            acc = accuracy_score(pred, y_data)
            eval_acc.append(acc)
    current_eval_acc = np.mean(eval_acc)
    if current_eval_acc > best_eval_acc:
        best_eval_acc = current_eval_acc
        print('best_acc is: {}'.format(current_eval_acc))
        print('save model to: ' + save_path)
        torch.save(model.state_dict(),save_path)

torch.save(model.state_dict(),train_parameters["checkpoints"]+"/"+"save_dir_final.ckpt")
draw_process("trainning loss","red",Iters,total_loss,"trainning loss")
draw_process("trainning acc","green",Iters,total_acc,"trainning acc")

## 6. 模型评估

- 读取训练的模型参数，在验证集上进行预测，使用准确率 accuracy 作为评价标准。

In [ ]:
model_state_dict = torch.load(train_parameters["checkpoints"]+"/"+"save_eval_best.ckpt", map_location=torch.device('cpu'))
model_eval = MyCNN(train_parameters['class_dim'])
model_eval.load_state_dict(model_state_dict)
model_eval.eval()
accs = []

for _, data in enumerate(eval_loader):
    x_data = data[0]
    y_data = data[1].view(-1)
    predicts = model_eval(x_data)
    pred = predicts.argmax(axis=1)
    acc = accuracy_score(pred, y_data)
    accs.append(acc)
print('模型在验证集上的准确率为：',np.mean(accs))

## 7. 模型改进

本项目分类数目多达 200 种，准确率较低，在验证集上准确率最高达到了 20%。

本项目训练过程中存在两个问题：
1.   训练时间过长，收敛较慢
2.   训练准确率较低，提升较慢

针对这两个问题，可以进行如下方面的改进：


1.   动态调整学习率，先设置较大的学习率，加快其收敛，后随迭代进行减小学习率以找到最优解。
2.   数据集中分类种类较多，数据量不足，可以收集更多数据，使用更复杂的数据增强，对图像进行亮度，对比度和角度的调整，增大数据量。
3.   使用残差结构以增加模型层数，提高模型特征提取能力。
4.   使用注意力机制提高模型对关键区域的关注度，提取关键特征增加识别准确度。




